In [ ]:
# немного посоветовался с нейронкой
class Vector():
    def __init__(self, x, y, z):
        assert isinstance(x, (int, float))
        assert isinstance(y, (int, float))
        assert isinstance(z, (int, float))
        self.x = x
        self.y = y
        self.z = z

    def __abs__(self):
        return (self.x ** 2 + self.y ** 2 + self.z ** 2) ** 0.5

    def __add__(self, other):
        if isinstance(other, Vector):
            return Vector(
                self.x + other.x, 
                self.y + other.y, 
                self.z + other.z
            )

    def __sub__(self, other):
        if isinstance(other, Vector):
            return Vector(
                self.x - other.x,
                self.y - other.y, 
                self.z - other.z
            )

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return Vector(
                self.x * other,
                self.y * other,
                self.z * other
            )
        if isinstance(other, Vector):
            return (
                self.x * other.x
                + self.y * other.y
                + self.z * other.z
            )

    def __rmul__(self, other):
        if isinstance(other, (int, float)):
            return Vector(
                self.x * other, 
                self.y * other, 
                self.z * other
            )


def center_of_mass(vectors, masses):
    mass = sum(masses)
    r = Vector(0, 0, 0)
    for i in range(len(vectors)):
        r += masses[i] * vectors[i]
    return r * (1 / mass)


def triangle(a, b, c):
    ab = b - a
    ac = c - a
    return ((ab * ab) * (ac * ac) - (ab * ac) ** 2) ** 0.5 / 2


def max_triangle(vectors):
    max1 = 0
    n = len(vectors)
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                p = triangle(vectors[i], vectors[j], vectors[k])
                if p > max1:
                    max1 = p
    return max1


In [ ]:
# нейронка помогла
from random import randrange as rnd, choice
from tkinter import *
import math
 
import time
root = Tk()
fr = Frame(root)
root.geometry('800x600')
canv = Canvas(root, bg='white')
canv.pack(fill=BOTH, expand=1)


class Ball():
    """ Класс Ball описывает мяч. """

    def __init__(self,x=40,y=450):
        """ Конструктор класса Ball
        Args:
        x - начальное положение мяча по горизонтали
        y - начальное положение мяча по вертикали
        """
        self.x = x
        self.y = y
        self.r = 10
        self.vx = 0
        self.vy = 0
        self.color = choice(['blue','green','red','brown'])
        self.id = canv.create_oval(self.x-self.r, self.y-self.r, self.x+self.r, self.y+self.r, fill=self.color)
        self.live = 30

    def set_coords(self):
        canv.coords(self.id, self.x-self.r, self.y-self.r, self.x+self.r, self.y+self.r)

    def move(self):
        """ Метод move описывает перемещение мяча за один кадр перерисовки. То есть, обновляет значения 
        self.x и self.y с учетом скоростей self.vx и self.vy, силы гравитации, действующей на мяч,
            и стен по краям окна (размер окна 800х600).
        """
        self.x += self.vx
        self.y += self.vy
        self.vy += 0.5
        if self.y + self.r >= 600:
            self.y = 600 - self.r
            self.vy = -self.vy * 0.8
            self.vx *= 0.8
        self.set_coords()

    def hittest(self,ob):
        """ Функция проверяет сталкивалкивается ли данный обьект с целью, описываемой в обьекте ob.

        Args:
            ob: Обьект, с которым проверяется столкновение.
        Returns:
            Возвращает True в случае столкновения мяча и цели. В противном случае возвращает False.
        """
        return ((self.x - ob.x) ** 2 +
            (self.y - ob.y) ** 2 <=
            (self.r + ob.r) ** 2)

class gun():
    """ Класс gun описывает пушку. """
    def __init__(self,x=40,y=450):
        self.f2_power = 10
        self.f2_on = 0
        self.an = 1
        self.id = canv.create_line(
            20, 450,
            20 + 30 * math.cos(self.an),
            450 + 30 * math.sin(self.an),
            width=7
        )
         
    def fire2_start(self,event):
        self.f2_on = 1
 
    def fire2_end(self,event):
        """ Выстрел мячом происходит при отпускании кнопки мыши.
        Начальные значения компонент скорости мяча vx и vy зависят от положения мыши.
        """
        global balls, bullet
        for b in balls:
            canv.delete(b.id)
        balls.clear()
        bullet += 1
        new_ball = Ball()
        new_ball.r += 5
        self.an = math.atan((event.y-new_ball.y)/(event.x-new_ball.x))
        new_ball.vx = self.f2_power*math.cos(self.an)
        new_ball.vy = -self.f2_power*math.sin(self.an)
        balls += [new_ball]
        self.f2_on = 0
        self.f2_power = 10
 
 
    def targetting (self,event=0):
        """ Прицеливание. Зависит от положения мыши.
        """
        if event:
            self.an = math.atan((event.y-450)/(event.x-20))    
        if self.f2_on:
            canv.itemconfig(self.id,fill = 'orange')
        else:
            canv.itemconfig(self.id,fill = 'black')
        canv.coords(self.id, 20, 450, 20 + max(self.f2_power, 20) * math.cos(self.an), 450 + max(self.f2_power, 20) * math.sin(self.an))
         

    def power_up(self):
        if self.f2_on:
            if self.f2_power < 100:
                self.f2_power += 1
            canv.itemconfig(self.id,fill = 'orange')
        else:
            canv.itemconfig(self.id,fill = 'black')
        
class target():
    """ Класс target описывает цель. """ 
    def __init__(self,x=40,y=450):
        self.points = 0
        self.live = 1
        self.id = canv.create_oval(0, 0, 0, 0)
        self.id_points = canv.create_text(30, 30, text = self.points, font = '28')
        self.new_target()
         
    def new_target(self):
        """ Инициализация новой цели. """
        x = self.x = rnd(600,780)
        y = self.y = rnd(300,550)
        r = self.r = rnd(2,50)
        color = self.color = 'red'
        canv.coords(self.id, x-r,y-r,x+r,y+r)
        canv.itemconfig(self.id, fill = color)
         
    def hit(self,points = 1):
        """ Попадание шарика в цель. """
        canv.coords(self.id,-10,-10,-10,-10)
        self.points += points
        canv.itemconfig(self.id_points, text = self.points)


t1 = target()
screen1 = canv.create_text(400,300, text = '',font = '28')
g1 = gun()
bullet = 0
balls = []



def new_game(event=''):
    global gun, t1, screen1, balls, bullet
    t1.new_target()
    bullet = 0
    balls = []
    canv.bind('<Button-1>', g1.fire2_start)
    canv.bind('<ButtonRelease-1>', g1.fire2_end)
    canv.bind('<Motion>', g1.targetting)
 
    z = 0.03
    t1.live = 1
    while t1.live or balls:
        for b in balls:
            b.move()
            if b.hittest(t1) and t1.live:
                t1.live = 0
                t1.hit()
                canv.bind('<Button-1>', '')
                canv.bind('<ButtonRelease-1>', '')
                canv.itemconfig(screen1, text = 'Вы уничтожили цель за ' + str(bullet) + ' выстрелов')
        if not t1.live:
            for b in balls:
                canv.delete(b.id)
            balls.clear()
        canv.update()
        time.sleep(0.03)
        g1.targetting()
        g1.power_up()
    canv.itemconfig(screen1, text = '')
    canv.delete(gun)
    root.after(750,new_game)

new_game()   
 
root.mainloop()


TclError: invalid command name ".!canvas"

In [ ]:
a = [int(i) for i in input().split()]
flag = True
for i in range(1, len(a)):
    if a[i] < a[(i - 1) // 2]:
        flag = False
        print('0')
        break
if flag:
    print('1')


In [ ]:
import random


def quick_sort(A):
    if len(A) <= 1:
        return A
    else:
        q = random.choice(A)
        L = []
        M = []
        R = []
        for elem in A:
            if elem < q:
                L.append(elem) 
            elif elem > q: 
                R.append(elem) 
            else: 
                M.append(elem)
        return quick_sort(L) + M + quick_sort(R)


f = [int(i) for i in input().split]
p = [int(i) for i in input().split]
m = [int(i) for i in input().split]
r = list((set(m) & set(p)) - set(f))
quick_sort(r)


TypeError: 'builtin_function_or_method' object is not iterable

In [ ]:
# посоветовался с нейронкой насчет оптимизации
# (в данной задаче подходит и словарь и 2 массива, но при большем количестве символов словарь выгоднее)
import random
v = {}
n = int(input())
for i in range(n):
    f = input()
    if f in v:
        v[f] += 1
    else:
        v[f] = 1
a = list(v.items())


def quick_sort(A):
    if len(A) <= 1:
        return A
    else:
        q = random.choice(A)[1]
        L = []
        M = []
        R = []
        for elem in A:
            if elem[1] < q:
                R.append(elem) 
            elif elem[1] > q: 
                L.append(elem) 
            else: 
                M.append(elem)
        return quick_sort(L) + M + quick_sort(R)


a = quick_sort(a)
for film, c in a:
    print(film, c)


ValueError: invalid literal for int() with base 10: ''

In [ ]:
# немного помогла нейронка
import sys 
  
class MinHeap: 
  
    def __init__(self, maxsize): 
        self.maxsize = maxsize 
        self.size = 0
        self.Heap = [0] * (self.maxsize + 1) 
        self.Heap[0] = -1 * 1e10 # or sys.maxsize
        self.FRONT = 1
  
    def parent(self, pos): 
        return pos // 2
  
    def leftChild(self, pos): 
        return 2 * pos 
  
    def rightChild(self, pos): 
        return (2 * pos) + 1
  
    def isLeaf(self, pos): 
        return pos * 2 > self.size 
  
    def swap(self, fpos, spos): 
        self.Heap[fpos], self.Heap[spos] = self.Heap[spos], self.Heap[fpos] 
  
    def minHeap(self):
        for pos in range(self.size // 2, 0, -1): 
            self.minHeapify(pos) 
    
    def minHeapify(self, pos):
        if not self.isLeaf(pos): 
            if (self.Heap[pos] > self.Heap[self.leftChild(pos)] or 
               self.Heap[pos] > self.Heap[self.rightChild(pos)]):
                if self.Heap[self.leftChild(pos)] < self.Heap[self.rightChild(pos)]: 
                    self.swap(pos, self.leftChild(pos)) 
                    self.minHeapify(self.leftChild(pos)) 
                else: 
                    self.swap(pos, self.rightChild(pos)) 
                    self.minHeapify(self.rightChild(pos)) 
  
    def insert(self, element): 
        if self.size >= self.maxsize: 
            return
        self.size += 1
        self.Heap[self.size] = element 
        current = self.size 
        while self.Heap[current] < self.Heap[self.parent(current)]: 
            self.swap(current, self.parent(current)) 
            current = self.parent(current) 
  
    def print_heap(self): 
        for i in range(1, (self.size // 2) + 1): 
            print(" PARENT : " + str(self.Heap[i]) + " LEFT CHILD : " + 
                                str(self.Heap[2 * i]) + " RIGHT CHILD : " +
                                str(self.Heap[2 * i + 1])) 
  
    
    def remove_head(self):
        popped = self.Heap[self.FRONT] 
        self.Heap[self.FRONT] = self.Heap[self.size] 
        self.size -= 1
        self.minHeapify(self.FRONT) 
        return popped


a = input().split()
minHeap = MinHeap(len(a))
for i in range(len(a)):
    a[i] = int(a[i])
    minHeap.insert(a[i])
minHeap.minHeap()
r = []
while minHeap.size > 0:
    r.append(minHeap.remove_head())
print(*r)
